# Supermarket Sales Analysis Using Python and Data Analytics

**AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026 | BharatCares**

### Problem Statement
Analyze supermarket sales transactions to understand product, branch, category, customer, payment, rating, and time-based performance and convert the findings into practical business insights.

### Dataset
The project uses the provided 500-row supermarket sales CSV. The original dataset is kept unchanged.

### Objectives
- Inspect and clean the data.
- Verify the sales calculation.
- Analyze products, branches, categories, customers, payments, ratings, and time trends.
- Visualize important findings.
- Provide evidence-based business recommendations.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

file_path = "SUPER MARKET DATA - supermarket_sales_500_rows.csv"
df = pd.read_csv(file_path)
df["Date"] = pd.to_datetime(df["Date"])
df.head()

In [ ]:
print("Rows and columns:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nDescriptive statistics:")
display(df.describe(include="all"))

## Data Validation and Sales Verification

In [ ]:
df["Calculated_Sales"] = df["Quantity"] * df["Unit Price"]
df["Sales_Difference"] = df["Calculated_Sales"] - df["Sales"]

print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("Sales mismatches (> ₹0.01):", int((df["Sales_Difference"].abs() > 0.01).sum()))
print("Maximum absolute sales difference:", df["Sales_Difference"].abs().max())

validation = pd.DataFrame({
    "Metric": ["Rows", "Columns", "Total Sales", "Average Transaction", "Total Quantity",
               "Average Quantity", "Average Unit Price", "Average Rating"],
    "Value": [len(df), len(df.columns)-2, df["Sales"].sum(), df["Sales"].mean(),
              df["Quantity"].sum(), df["Quantity"].mean(), df["Unit Price"].mean(), df["Rating"].mean()]
})
display(validation.round(2))

## Product Analysis

In [ ]:
product_summary = df.groupby("Product").agg(
    Total_Sales=("Sales","sum"),
    Transactions=("Sales","count"),
    Total_Quantity=("Quantity","sum"),
    Average_Sales=("Sales","mean")
).sort_values("Total_Sales", ascending=False)
display(product_summary.round(2))
print("Highest-sales product:", product_summary.index[0], "₹", round(product_summary.iloc[0]["Total_Sales"],2))
print("Highest-quantity product:", product_summary["Total_Quantity"].idxmax(), product_summary["Total_Quantity"].max())

In [ ]:
product_summary["Total_Sales"].head(10).sort_values().plot(kind="barh", figsize=(9,5))
plt.title("Top 10 Products by Sales"); plt.xlabel("Sales (₹)"); plt.ylabel("Product")
plt.tight_layout(); plt.show()

## Branch and City Analysis

In [ ]:
branch_summary = df.groupby(["Branch","City"]).agg(
    Total_Sales=("Sales","sum"),
    Transactions=("Sales","count"),
    Total_Quantity=("Quantity","sum"),
    Average_Sales=("Sales","mean"),
    Average_Rating=("Rating","mean")
).sort_values("Total_Sales", ascending=False)
display(branch_summary.round(2))

In [ ]:
branch_summary["Total_Sales"].plot(kind="bar", figsize=(8,5))
plt.title("Sales by Branch"); plt.xlabel("Branch"); plt.ylabel("Sales (₹)")
plt.tight_layout(); plt.show()

## Category Analysis

In [ ]:
category_summary = df.groupby("Category").agg(
    Total_Sales=("Sales","sum"),
    Transactions=("Sales","count"),
    Total_Quantity=("Quantity","sum"),
    Average_Sales=("Sales","mean")
).sort_values("Total_Sales", ascending=False)
display(category_summary.round(2))
print("Highest-sales category:", category_summary.index[0], "₹", round(category_summary.iloc[0]["Total_Sales"],2))

In [ ]:
category_summary["Total_Sales"].plot(kind="bar", figsize=(9,5))
plt.title("Sales by Category"); plt.xlabel("Category"); plt.ylabel("Sales (₹)")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

## Payment Method Analysis

In [ ]:
payment_summary = df.groupby("Payment").agg(
    Transactions=("Sales","count"),
    Total_Sales=("Sales","sum"),
    Average_Sales=("Sales","mean")
).sort_values("Transactions", ascending=False)
payment_summary["Transaction_Percentage"] = payment_summary["Transactions"] / len(df) * 100
display(payment_summary.round(2))

In [ ]:
payment_summary["Transactions"].plot(kind="bar", figsize=(8,5))
plt.title("Transactions by Payment Method"); plt.xlabel("Payment"); plt.ylabel("Transactions")
plt.tight_layout(); plt.show()

## Customer Type, Gender and Rating Analysis

In [ ]:
customer_summary = df.groupby("Customer Type").agg(
    Transactions=("Sales","count"),
    Total_Sales=("Sales","sum"),
    Average_Sales=("Sales","mean"),
    Total_Quantity=("Quantity","sum"),
    Average_Rating=("Rating","mean")
)
display(customer_summary.round(2))

gender_summary = df.groupby("Gender").agg(
    Transactions=("Sales","count"),
    Total_Sales=("Sales","sum"),
    Average_Sales=("Sales","mean"),
    Total_Quantity=("Quantity","sum"),
    Average_Rating=("Rating","mean")
)
display(gender_summary.round(2))

print("Average Member transaction:", round(customer_summary.loc["Member","Average_Sales"],2))
print("Average Normal transaction:", round(customer_summary.loc["Normal","Average_Sales"],2))

In [ ]:
rating_counts = df["Rating"].value_counts().sort_index()
rating_counts.plot(kind="bar", figsize=(9,5))
plt.title("Customer Rating Distribution"); plt.xlabel("Rating"); plt.ylabel("Transactions")
plt.tight_layout(); plt.show()

print("Average rating:", round(df["Rating"].mean(),2))
print("Ratings below 3:", int((df["Rating"] < 3).sum()))
print("Ratings 3 or above:", int((df["Rating"] >= 3).sum()))
print("Ratings 4 or above:", int((df["Rating"] >= 4).sum()))

## Time-Based Analysis

In [ ]:
df["Month"] = df["Date"].dt.to_period("M").astype(str)
df["Day_of_Week"] = df["Date"].dt.day_name()

monthly_sales = df.groupby("Month")["Sales"].sum().sort_values(ascending=False)
daily_sales = df.groupby("Day_of_Week")["Sales"].sum().sort_values(ascending=False)
date_sales = df.groupby("Date")["Sales"].sum().sort_values(ascending=False)

display(monthly_sales.round(2))
display(daily_sales.round(2))
print("Highest-sales month:", monthly_sales.index[0], "₹", round(monthly_sales.iloc[0],2))
print("Best day of week:", daily_sales.index[0], "₹", round(daily_sales.iloc[0],2))
print("Highest-sales date:", date_sales.index[0].date(), "₹", round(date_sales.iloc[0],2))

In [ ]:
monthly_sales.sort_index().plot(kind="line", marker="o", figsize=(9,5))
plt.title("Monthly Sales Trend"); plt.xlabel("Month"); plt.ylabel("Sales (₹)")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

daily_sales.plot(kind="bar", figsize=(9,5))
plt.title("Sales by Day of Week"); plt.xlabel("Day"); plt.ylabel("Sales (₹)")
plt.tight_layout(); plt.show()

## Key Business Questions and Answers

In [ ]:
print("1. Highest-sales product:", product_sales.index[0], "₹", round(product_sales.iloc[0],2))
print("2. Best branch:", branch_sales.index[0][0], "-", branch_sales.index[0][1], "₹", round(branch_sales.iloc[0],2))
print("3. Highest-sales category:", cat_sales.index[0], "₹", round(cat_sales.iloc[0],2))
print("4. Most-used payment method:", pay_count.index[0], "-", int(pay_count.iloc[0]), "transactions")
print("5. Do Members spend more?", "No" if cust_avg["Member"] < cust_avg["Normal"] else "Yes")
print("   Member average:", round(cust_avg["Member"],2), "| Normal average:", round(cust_avg["Normal"],2))
print("6. Average customer rating:", round(avg_rating,2), "out of 5")
print("7. Highest-quantity product:", product_qty.index[0], "-", int(product_qty.iloc[0]), "units")
print("8. Highest average transaction branch:", branch_avg.index[0], "₹", round(branch_avg.iloc[0],2))
print("9. Payment method with highest sales:", pay_sales.index[0], "₹", round(pay_sales.iloc[0],2))
print("10. Highest-sales month:", month_sales.index[0], "₹", round(month_sales.iloc[0],2))
print("11. Best day of week:", dow_sales.index[0], "₹", round(dow_sales.iloc[0],2))

## Business Insights and Recommendations

- Cheese has the highest total sales among products, so its availability should be monitored closely.
- Branch C in Mumbai has the highest branch sales; management can study its product mix, customer flow, and operating practices.
- Beverages is the highest-sales category and can receive focused inventory and promotional planning.
- UPI is the most-used payment method, so reliable digital payment support should continue.
- Normal customers have a higher average transaction value than Members in this dataset; membership offers should therefore be evaluated using actual spending behavior rather than assuming Members spend more.
- The overall rating is close to 4 out of 5, indicating generally positive customer feedback while still leaving room for service improvement.
- Monday has the highest sales among weekdays in this dataset, while April is the highest-sales month. Staffing and promotions can be planned with these observed patterns in mind.

These are observations and practical implications from this dataset; they do not establish causation.

## Conclusion

This project demonstrates how supermarket transaction data can be transformed into actionable business information using Python. The analysis covers products, branches, categories, payments, customer types, gender, ratings, and time trends. The findings can support inventory planning, branch performance review, payment-channel support, customer strategy, and service improvement.

### Limitations
The dataset contains 500 transactions and represents a limited observation period. The analysis is descriptive and does not establish causal relationships. Future work could include forecasting, customer segmentation, and predictive modelling if a sufficiently long historical dataset is available.